# Remapeo de identificación de daños en vigas

## 📋 Resumen Ejecutivo

Este notebook implementa un **sistema de remapeo inteligente** para mejorar la detección de daños en vigas (beams) procesadas por un Algoritmo Genético (AG).

## 🎯 Objetivo Principal

Las vigas son elementos estructurales **difícilmente identificables de forma directa** por el AG. Sin embargo, cuando el AG detecta daño en **nodos cercanos** a una viga inicialmente dañada, es razonable inferir que esa viga también está dañada. Este notebook implementa esa lógica de **detección parcial por proximidad**.

## 🔍 Problema que Resuelve

**Situación inicial:**
- El AG busca elementos dañados en una estructura
- Muchas vigas con daño real NO son detectadas directamente (`DeteccionOK = False`)
- PERO el AG SÍ detecta daño en nodos estructurales muy cercanos a esas vigas

**Solución implementada:**
- Si una viga no fue detectada directamente, pero hay daño en sus nodos cercanos → Asignar **detección parcial** (`DeteccionOK = 0.5`)
- Esto crea una categoría intermedia entre "no detectado" y "detectado"

## 📊 Sistema de Clasificación Final

| Valor | Significado | Descripción |
|-------|------------|-------------|
| `0` | No detectado | El AG no encontró la viga ni nodos cercanos con daño |
| `0.5` | **Detección parcial** | El AG detectó daño en nodos cercanos a la viga |
| `1` | Detectado | El AG detectó la viga directamente |

## 🗂️ Archivos de Entrada

1. **`todos_los_resultados_csv.csv`**: Dataset principal con resultados de múltiples ejecuciones del AG
2. **`nodos_beams.csv`**: Mapeo de cada viga con sus 2 nodos estructurales más cercanos
3. **`salida_csvs/*.csv`**: Resultados individuales del AG (ID_0001.csv, ID_0002.csv, etc.) indicando qué nodos tienen daño

## 📤 Archivo de Salida

- **`todos_los_resultados_csv_remapeado.csv`**: Dataset modificado con detecciones parciales incluidas

## 🔧 Proceso Completo

1. **Carga de datos**: Dataset principal y archivos auxiliares
2. **Mapeo de nodos cercanos**: Identificación de nodos próximos a cada viga
3. **Análisis de CSVs del AG**: Verificación de detecciones en nodos cercanos
4. **Remapeo**: Cambio de `False` → `0.5` para vigas con detección cercana
5. **Normalización**: Conversión de booleanos a valores numéricos
6. **Exportación**: Guardado del dataset remapeado

---

_Dependencias correspondientes_

In [1]:
# Librerías para manejo de datos
import pandas as pd
import numpy as np

# Librerías para cálculo de distancias y operaciones espaciales
from scipy.spatial.distance import cdist, euclidean
from scipy.spatial import KDTree

# Librerías para manejo de archivos y directorios
import os
import sys
import glob
from pathlib import Path
import shutil  # Para copiar/mover archivos si necesitas respaldos

# Para navegar directorios relativos desde el notebook
import json  # Por si necesitas leer/escribir configuraciones

# Librerías para visualización (opcional, si necesitas graficar)
import matplotlib.pyplot as plt
import seaborn as sns

# Para copias de seguridad y manejo de datos
import copy
from datetime import datetime

# Para iterar y filtrar de manera eficiente
from itertools import product, combinations
import warnings
warnings.filterwarnings('ignore')  # Para suprimir warnings innecesarios

_Carga de todos_los_resultados_csv: dicho archivo será el que modificaremos._

In [3]:
# Cargar el archivo CSV desde ../jupyter_notebooks/
csv_path = Path('..') / 'jupyter_notebooks' / 'todos_los_resultados.csv'
df = pd.read_csv(csv_path)

# Convertir DetectionOK de VERDADERO/FALSO a 1/0
df['DetectionOK'] = df['DetectionOK'].map({'VERDADERO': 1, 'FALSO': 0})

print(f"Archivo cargado: {csv_path}")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print("\nPrimeras filas del DataFrame:")
df.head()

Archivo cargado: ../jupyter_notebooks/todos_los_resultados.csv
Dimensiones: 2160 filas x 12 columnas

Primeras filas del DataFrame:


,ID,Element,Percentage,Time_seconds,Final_Objective,DetectionOK,AvgDispersion,StdDispersion,MeanAbsDispersion,N_FalsePositives,Type_of_element_to_search,Story
0,1,1,5,4178.298416,0.312917,1,10.264056,9.668608,10.264056,0,inclined_leg,Story 1
1,2,1,10,4234.230792,0.156595,1,7.835071,6.578796,7.835071,0,inclined_leg,Story 1
2,3,1,15,4288.113483,0.140108,1,7.077523,6.467719,7.077523,0,inclined_leg,Story 1
3,4,1,20,4353.540449,0.136951,1,6.742003,6.516048,6.742003,0,inclined_leg,Story 1
4,5,1,25,4418.229896,0.134341,1,6.527849,6.542074,6.527849,0,inclined_leg,Story 1


_Copiado del dataframe_

In [4]:
# Crear una copia del DataFrame original para trabajar sin modificar el archivo fuente
df_modificado = df.copy()

print("DataFrame copiado exitosamente.")
print(f"df original: {df.shape}")
print(f"df_modificado: {df_modificado.shape}")
print("\nAhora puedes trabajar con 'df_modificado' sin afectar 'df' ni el archivo original.")

DataFrame copiado exitosamente.
df original: (2160, 12)
df_modificado: (2160, 12)

Ahora puedes trabajar con 'df_modificado' sin afectar 'df' ni el archivo original.


# Conversión de daño parcial a los beam si el nodo con daño identificado se encuentra cercano al beam inicialmente con daño

## Bloque 1: Cargar nodos_beams.csv y explorarlo

In [5]:
# Cargar el archivo nodos_beams.csv
nodos_beams_path = Path('.') / 'nodos_beams.csv'
df_nodos_beams = pd.read_csv(nodos_beams_path)

print(f"Archivo cargado: {nodos_beams_path}")
print(f"Dimensiones: {df_nodos_beams.shape[0]} filas x {df_nodos_beams.shape[1]} columnas")
print(f"\nColumnas: {list(df_nodos_beams.columns)}")
print("\nPrimeras filas:")
df_nodos_beams

Archivo cargado: nodos_beams.csv
Dimensiones: 16 filas x 5 columnas

Columnas: ['beam', 'nodo_i', 'nodo_j', 'nodo_cercano_1', 'nodo_cercano_2']

Primeras filas:


,beam,nodo_i,nodo_j,nodo_cercano_1,nodo_cercano_2
0,96,36,35,31,39
1,95,34,35,38,30
2,94,33,34,37,29
3,93,33,36,40,32
4,69,28,27,31,24
5,72,26,27,30,23
6,71,25,26,29,22
7,70,25,28,21,32
8,45,17,18,13,22
9,46,18,19,23,14


## Bloque 2: Listar y explorar los CSVs en ./salida_csvs/

In [6]:
# Listar todos los archivos CSV en el directorio salida_csvs
salida_csvs_path = Path('.') / 'salida_csvs'
csv_files = sorted(salida_csvs_path.glob('*.csv'))

print(f"Directorio: {salida_csvs_path}")
print(f"Total de archivos CSV encontrados: {len(csv_files)}")
print("\nPrimeros 10 archivos:")
for i, file in enumerate(csv_files[:10], 1):
    print(f"  {i}. {file.name}")

Directorio: salida_csvs
Total de archivos CSV encontrados: 2160

Primeros 10 archivos:
  1. ID_0001.csv
  2. ID_0002.csv
  3. ID_0003.csv
  4. ID_0004.csv
  5. ID_0005.csv
  6. ID_0006.csv
  7. ID_0007.csv
  8. ID_0008.csv
  9. ID_0009.csv
  10. ID_0010.csv


## Bloque 3: Cargar un CSV de ejemplo y ver su estructura

In [7]:
# Cargar un CSV de ejemplo para ver su estructura
if len(csv_files) > 0:
    ejemplo_csv = csv_files[0]
    df_ejemplo = pd.read_csv(ejemplo_csv)
    
    print(f"Archivo de ejemplo: {ejemplo_csv.name}")
    print(f"Dimensiones: {df_ejemplo.shape}")
    print(f"\nColumnas: {list(df_ejemplo.columns)}")
    print("\nPrimeras 15 filas:")
    print(df_ejemplo.head(15))
    
    # Ver cuántos nodos tienen Estado = "Daño"
    nodos_con_dano = df_ejemplo[df_ejemplo['Estado'] == 'Daño']
    print(f"\n\nNodos detectados con daño: {len(nodos_con_dano)}")
    if len(nodos_con_dano) > 0:
        print("\nNodos con daño:")
        print(nodos_con_dano[['Numero_de_nodo', 'Valor_de_daño_normalizado', 'Estado']])
else:
    print("No se encontraron archivos CSV en salida_csvs/")

Archivo de ejemplo: ID_0001.csv
Dimensiones: (48, 3)

Columnas: ['Numero_de_nodo', 'Valor_de_daño_normalizado', 'Estado']

Primeras 15 filas:
    Numero_de_nodo  Valor_de_daño_normalizado Estado
0                5                   9.168741      -
1                6                   2.581235      -
2                7                   7.879373      -
3                8                   6.314645      -
4                9                  12.270594      -
5               10                  17.020868      -
6               11                   7.790110      -
7               12                   5.492115      -
8               13                  15.310780      -
9               14                   2.690071      -
10              15                   2.917306      -
11              16                   4.530742      -
12              17                  30.328585      -
13              18                   4.667297      -
14              19                   2.929966      -


Nodos de

## Bloque 4: Función para verificar si un beam tiene detección cercana

In [8]:
def verificar_deteccion_cercana(beam_id, df_nodos_beams, csv_file):
    """
    Verifica si un beam tiene detección en nodos cercanos en un archivo CSV del AG.
    
    Parámetros:
    - beam_id: ID del beam a verificar
    - df_nodos_beams: DataFrame con información de nodos de beams
    - csv_file: Ruta al archivo CSV de resultados del AG
    
    Retorna:
    - True si se detectó daño en nodo_cercano_1 o nodo_cercano_2
    - False en caso contrario
    """
    # Obtener información del beam
    beam_info = df_nodos_beams[df_nodos_beams['beam'] == beam_id]
    
    if beam_info.empty:
        return False
    
    # Obtener los nodos cercanos
    nodo_cercano_1 = beam_info['nodo_cercano_1'].values[0]
    nodo_cercano_2 = beam_info['nodo_cercano_2'].values[0]
    
    # Cargar el CSV del AG
    try:
        df_ag = pd.read_csv(csv_file)
        
        # Filtrar nodos con Estado = "Daño" (con ñ y D mayúscula)
        # Usar strip() para eliminar espacios en blanco
        df_ag['Estado'] = df_ag['Estado'].astype(str).str.strip()
        nodos_con_dano = df_ag[df_ag['Estado'] == 'Daño']['Numero_de_nodo'].values
        
        # Verificar si alguno de los nodos cercanos fue detectado
        if nodo_cercano_1 in nodos_con_dano or nodo_cercano_2 in nodos_con_dano:
            return True
    except Exception as e:
        print(f"Error al procesar {csv_file.name}: {e}")
        return False
    
    return False

# Prueba de la función con un ejemplo
print("Función creada exitosamente (con corrección para 'Daño').")
print("\nPrueba de la función con el primer beam y primer CSV:")
if len(csv_files) > 0 and len(df_nodos_beams) > 0:
    primer_beam = df_nodos_beams['beam'].iloc[0]
    resultado = verificar_deteccion_cercana(primer_beam, df_nodos_beams, csv_files[0])
    print(f"Beam: {primer_beam}")
    print(f"CSV: {csv_files[0].name}")
    print(f"¿Detección cercana?: {resultado}")
    
    # Debug: Verificar qué valores únicos tiene la columna Estado en el CSV
    df_test = pd.read_csv(csv_files[0])
    print(f"\nValores únicos en 'Estado': {df_test['Estado'].unique()}")
    print(f"Nodos con 'Daño': {df_test[df_test['Estado'] == 'Daño']['Numero_de_nodo'].values}")

Función creada exitosamente (con corrección para 'Daño').

Prueba de la función con el primer beam y primer CSV:
Beam: 96
CSV: ID_0001.csv
¿Detección cercana?: False

Valores únicos en 'Estado': ['-' 'Daño']
Nodos con 'Daño': [22 26]


## Bloque 5: Aplicar la función a todos los CSVs y actualizar df_modificado

In [9]:
# Aplicar remapeo: cambiar DetectionOK de 0 a 0.5 cuando hay detección cercana
print("Iniciando proceso de remapeo...")
print(f"Total de CSVs a procesar: {len(csv_files)}")
print(f"Total de filas en df_modificado: {len(df_modificado)}\n")

# Filtrar solo las filas que corresponden a beams
filas_beam = df_modificado[df_modificado['Type_of_element_to_search'] == 'beam']
print(f"Total de filas con Type_of_element_to_search = 'beam': {len(filas_beam)}\n")

# Contador de cambios
cambios_realizados = 0
filas_procesadas = 0

# Iterar sobre cada fila de beams en df_modificado
for idx, row in df_modificado.iterrows():
    # Solo procesar si es un beam
    if row['Type_of_element_to_search'] != 'beam':
        continue
    
    filas_procesadas += 1
    
    # Solo procesar filas donde DetectionOK es 0
    if row['DetectionOK'] == 0:
        
        # Usar 'Element' en lugar de 'Elemento' para obtener el beam_id
        beam_id = row['Element']
        
        # Para cada CSV, verificar si hay detección cercana
        deteccion_encontrada = False
        for csv_file in csv_files:
            if verificar_deteccion_cercana(beam_id, df_nodos_beams, csv_file):
                deteccion_encontrada = True
                break  # Si encontramos en un CSV, no necesitamos seguir buscando
        
        # Si se encontró detección cercana, cambiar a 0.5
        if deteccion_encontrada:
            df_modificado.at[idx, 'DetectionOK'] = 0.5
            cambios_realizados += 1
    
    # Mostrar progreso cada 50 filas
    if filas_procesadas % 50 == 0:
        print(f"Procesadas {filas_procesadas} filas de beams... Cambios realizados: {cambios_realizados}")

print(f"\n✅ Proceso completado!")
print(f"Total de filas de beams procesadas: {filas_procesadas}")
print(f"Total de cambios realizados (0 → 0.5): {cambios_realizados}")

Iniciando proceso de remapeo...
Total de CSVs a procesar: 2160
Total de filas en df_modificado: 2160

Total de filas con Type_of_element_to_search = 'beam': 360

Procesadas 50 filas de beams... Cambios realizados: 50
Procesadas 50 filas de beams... Cambios realizados: 50
Procesadas 100 filas de beams... Cambios realizados: 100
Procesadas 100 filas de beams... Cambios realizados: 100
Procesadas 150 filas de beams... Cambios realizados: 150
Procesadas 150 filas de beams... Cambios realizados: 150
Procesadas 200 filas de beams... Cambios realizados: 200
Procesadas 200 filas de beams... Cambios realizados: 200
Procesadas 250 filas de beams... Cambios realizados: 250
Procesadas 250 filas de beams... Cambios realizados: 250
Procesadas 300 filas de beams... Cambios realizados: 288
Procesadas 300 filas de beams... Cambios realizados: 288
Procesadas 350 filas de beams... Cambios realizados: 288
Procesadas 350 filas de beams... Cambios realizados: 288

✅ Proceso completado!
Total de filas de bea

## Bloque 6: Verificar cambios y guardar resultado

In [10]:
# Verificar los cambios realizados
print("=== VERIFICACIÓN DE CAMBIOS ===\n")

# Comparar valores de DetectionOK
print("Distribución de valores en df original:")
print(df['DetectionOK'].value_counts())

print("\nDistribución de valores en df_modificado:")
print(df_modificado['DetectionOK'].value_counts())

# Ver algunas filas que cambiaron a 0.5
filas_con_05 = df_modificado[df_modificado['DetectionOK'] == 0.5]
print(f"\nTotal de filas con DetectionOK = 0.5: {len(filas_con_05)}")

if len(filas_con_05) > 0:
    print("\nPrimeras 10 filas con DetectionOK = 0.5:")
    print(filas_con_05.head(10))

=== VERIFICACIÓN DE CAMBIOS ===

Distribución de valores en df original:
DetectionOK
1    1724
0     436
Name: count, dtype: int64

Distribución de valores en df_modificado:
DetectionOK
1.0    1724
0.5     288
0.0     148
Name: count, dtype: int64

Total de filas con DetectionOK = 0.5: 288

Primeras 10 filas con DetectionOK = 0.5:
      ID  Element  Percentage  Time_seconds  Final_Objective  DetectionOK  \
360  361       21           5   20696.06673         1.863363          0.5   
361  362       21          10   20731.93542         1.858548          0.5   
362  363       21          15   20783.05247         1.853382          0.5   
363  364       21          20   20822.14855         1.847747          0.5   
364  365       21          25   20874.73642         1.841772          0.5   
365  366       21          30   20912.73820         1.835157          0.5   
366  367       21          35   20961.48062         1.828225          0.5   
367  368       21          40   20992.79615        

## Bloque 7: Convertir True en 1 y False en 0

In [11]:
# Ya no necesitamos convertir booleanos porque DetectionOK ya es numérico (0/1)
# Solo verificamos que los valores sean correctos

print("=== VERIFICACIÓN DE VALORES FINALES ===\n")

print("Valores únicos en DetectionOK:")
print(df_modificado['DetectionOK'].unique())
print(f"\nTipo de datos: {df_modificado['DetectionOK'].dtype}")

print("\nDistribución final de valores:")
print(df_modificado['DetectionOK'].value_counts().sort_index())

print("\n✅ Valores finales:")
print("   • 0 → No detectado")
print("   • 0.5 → Detección cercana (remapeado)")
print("   • 1 → Detectado directamente")

=== VERIFICACIÓN DE VALORES FINALES ===

Valores únicos en DetectionOK:
[1.  0.5 0. ]

Tipo de datos: float64

Distribución final de valores:
DetectionOK
0.0     148
0.5     288
1.0    1724
Name: count, dtype: int64

✅ Valores finales:
   • 0 → No detectado
   • 0.5 → Detección cercana (remapeado)
   • 1 → Detectado directamente


## Bloque 8: Guardar DataFrame modificado

In [13]:
# Guardar el DataFrame modificado en CSV
output_path = Path('..') / 'jupyter_notebooks' / 'todos_los_resultados_csv_remapeado.csv'
df_modificado.to_csv(output_path, index=False)

print(f"✅ Archivo guardado exitosamente en:")
print(f"   {output_path}")
print(f"\nResumen final:")
print(f"   • Total de filas: {len(df_modificado)}")
print(f"   • Distribución DetectionOK:")
print(df_modificado['DetectionOK'].value_counts().sort_index())

✅ Archivo guardado exitosamente en:
   ../jupyter_notebooks/todos_los_resultados_csv_remapeado.csv

Resumen final:
   • Total de filas: 2160
   • Distribución DetectionOK:
DetectionOK
0.0     148
0.5     288
1.0    1724
Name: count, dtype: int64


---

# 📝 Conclusión y Justificación del Remapeo

## ✅ Trabajo Completado

Este notebook ha implementado exitosamente un **sistema de detección parcial por proximidad** para elementos tipo viga (Beam) en análisis estructurales procesados por Algoritmos Genéticos.

## 🔬 Fundamento Técnico

### ¿Por qué las vigas son difíciles de detectar directamente?

Las vigas son elementos estructurales **lineales** que conectan nodos. A diferencia de los nodos (puntos específicos), las vigas representan:
- Elementos continuos en el espacio
- Distribución de daño a lo largo de su longitud
- Comportamiento mecánico distribuido

Esta naturaleza hace que el AG tenga **mayor dificultad** para identificarlas directamente en comparación con elementos puntuales.

### Estrategia de Detección por Proximidad

**Hipótesis fundamental:**
> Si el AG detecta daño en los nodos estructurales inmediatamente adyacentes a una viga que inicialmente tenía daño, existe una **alta probabilidad** de que la viga también esté dañada, aunque no haya sido detectada directamente.

**Razones físicas:**
1. **Continuidad estructural**: El daño en una viga afecta los nodos que conecta
2. **Transferencia de cargas**: Los nodos cercanos reflejan el comportamiento de la viga
3. **Propagación de daño**: El deterioro estructural se manifiesta en elementos adyacentes

## 📈 Valor del Sistema de Detección Parcial (0.5)

La introducción del valor `0.5` permite:

| Aspecto | Ventaja |
|---------|---------|
| **Granularidad** | Distinguir entre "no hay evidencia" (0) y "hay evidencia indirecta" (0.5) |
| **Análisis estadístico** | Calcular tasas de detección ponderadas más realistas |
| **Evaluación del AG** | Reconocer detecciones cercanas como éxitos parciales |
| **Toma de decisiones** | Priorizar inspecciones en elementos con detección parcial |

## 🎯 Impacto en Resultados

**Antes del remapeo:**
- Muchas vigas marcadas como "no detectadas" (0) aunque hubiera evidencia cercana
- Subestimación del rendimiento real del AG
- Pérdida de información valiosa sobre proximidad de detecciones

**Después del remapeo:**
- Reconocimiento de detecciones indirectas (0.5)
- Evaluación más justa y completa del AG
- Mayor información para análisis posteriores

## 🔄 Aplicabilidad

Este enfoque es especialmente útil para:
- ✓ Evaluación de algoritmos de detección de daño estructural
- ✓ Análisis de sensibilidad en sistemas de monitoreo
- ✓ Validación de modelos de elementos finitos
- ✓ Priorización de inspecciones en estructuras reales

## 💾 Archivo Generado

El archivo **`todos_los_resultados_csv_remapeado.csv`** contiene:
- Todas las filas del dataset original
- Columna `DeteccionOK` actualizada con valores: `0`, `0.5`, o `1`
- Listo para análisis estadísticos y visualizaciones avanzadas

---

**Fecha de procesamiento:** Octubre 2025  
**Método:** Detección parcial por proximidad de nodos  
**Criterio:** Daño en `nodo_cercano_1` O `nodo_cercano_2` → DeteccionOK = 0.5

In [15]:
df_modificado

,ID,Element,Percentage,Time_seconds,Final_Objective,DetectionOK,AvgDispersion,StdDispersion,MeanAbsDispersion,N_FalsePositives,Type_of_element_to_search,Story
0,1,1,5,4178.298416,0.312917,1.0,10.264056,9.668608,10.264056,0,inclined_leg,Story 1
1,2,1,10,4234.230792,0.156595,1.0,7.835071,6.578796,7.835071,0,inclined_leg,Story 1
2,3,1,15,4288.113483,0.140108,1.0,7.077523,6.467719,7.077523,0,inclined_leg,Story 1
3,4,1,20,4353.540449,0.136951,1.0,6.742003,6.516048,6.742003,0,inclined_leg,Story 1
4,5,1,25,4418.229896,0.134341,1.0,6.527849,6.542074,6.527849,0,inclined_leg,Story 1
...,...,...,...,...,...,...,...,...,...,...,...,...
2155,2156,120,70,138778.439900,1.666667,0.0,23.970995,13.234665,23.970995,14,beam,Story 5
2156,2157,120,75,138868.615400,1.666667,0.0,21.949217,13.180450,21.949217,14,beam,Story 5
2157,2158,120,80,138951.163100,1.666667,0.0,26.630774,17.488262,26.882047,15,beam,Story 5
2158,2159,120,85,139042.948000,1.666667,0.0,20.975020,10.406626,20.975020,0,beam,Story 5
